# Sensitivity functions — S, T, PS, CS, and the loop under disturbance and noise

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/alx87grd/minilink/blob/main/examples/teaching/topics/classical_control/sensitivity_functions.ipynb)

A feedback loop does more than track a reference. It also decides what a **load disturbance** $w$ does to the output and what **measurement noise** $v$ does to the actuator. Four transfer functions tell the whole story, read with the [minilink](https://github.com/alx87grd/minilink) API:

- the plant $H(s)$, input force to measured position;
- the controller $C(s)$, error to force;
- an optional filter $F(s)$ on the measurement ($F = 1$ here);
- the loop gain $L(s) = C(s)\,H(s)\,F(s)$.

| function | formula | from → to | minilink |
| --- | --- | --- | --- |
| sensitivity | $S = \dfrac{1}{1 + L}$ | $r \to e$, and an output disturbance → $y$ | `sensitivity` |
| complementary sensitivity | $T = \dfrac{C H}{1 + L}$ | $r \to y$, and $v \to y$ (sign aside) | `complementary_sensitivity` |
| load sensitivity | $PS = \dfrac{H}{1 + L}$ | $w \to y$ | `load_sensitivity` |
| noise sensitivity | $CS = \dfrac{C F}{1 + L}$ | $v \to u$ | `noise_sensitivity` |

Each function takes the pieces by name, `sensitivity(plant=H, controller=C)`, and returns a `TransferFunction` whose ports carry the loop signals it maps (`r` to `e` for $S$), so `bode` and `plot_bode` read it like any other system and label the channel. Then `closed_loop(C, plant, w=True, v=True)` builds the same loop as a diagram with three inputs, $r$, $w$ and $v$, and three source blocks driving them check the predictions in simulation.

In [ ]:
# Local: minilink already installed. Colab: clone + path.
import sys

if "google.colab" in sys.modules:
    get_ipython().run_line_magic("matplotlib", "inline")
    get_ipython().system("git clone https://github.com/alx87grd/minilink")
    sys.path.insert(0, "/content/minilink")

In [ ]:
import numpy as np

from minilink import PID, DiagramSystem, Sine, SingleMass, Step, closed_loop
from minilink.analysis import (
    bode,
    complementary_sensitivity,
    frequency_response,
    load_sensitivity,
    noise_sensitivity,
    plot_bode,
    sensitivity,
)

BACKEND = "matplotlib"  # or "plotly" (hover; the one to use in Colab)
# BACKEND = "plotly"

REFERENCE = 0.2  # setpoint step r: m
LOAD_AMPLITUDE, LOAD_OMEGA = 1.0, 1.0  # load force w: N, rad/s
NOISE_AMPLITUDE, NOISE_OMEGA = 0.02, 50.0  # position noise v: m, rad/s

## 1. The pieces of the loop

The plant is a mass on a spring with a damper, $m\,\ddot x = F - k\,x - b\,\dot x$: one input, the force $F$, and one output, the position $x$, so $H(s) = \dfrac{1}{m s^2 + b s + k}$ with a resonance at $\sqrt{k/m} \approx 1.41$ rad/s. The controller is a PID with a filtered derivative, $C(s) = K_p + \dfrac{K_i}{s} + \dfrac{K_d\, s}{\tau s + 1}$, on the error $e = r - x$.

In [ ]:
plant = SingleMass()  # m = 1 kg, k = 2 N/m
plant.params["b"] = 0.5  # N·s/m, a little damping

C = PID(Kp=10.0, Ki=5.0, Kd=4.0, tau=0.05)  # error -> force

The four functions, from the pieces. The body of each one writes $L = C H F$ and then the formula of the table above.

In [ ]:
S = sensitivity(plant=plant, controller=C)
T = complementary_sensitivity(plant=plant, controller=C)
PS = load_sensitivity(plant=plant, controller=C)
CS = noise_sensitivity(plant=plant, controller=C)

## 2. Reading the four Bode diagrams

**Sensitivity $S$.** Below the crossover, $|L| \gg 1$ and $|S| \approx 1/|L|$ is small: the integral action makes $S(0) = 0$, so a constant disturbance on the output is fully rejected. Above the crossover, $|L| \ll 1$ and $|S| \to 1$: feedback no longer acts. In between, $|S|$ peaks just above 0 dB. That peak, $M_s = \max_\omega |S(j\omega)|$, is the inverse of the distance from $L(j\omega)$ to the critical point $-1$: here about +1.7 dB, $M_s \approx 1.2$, so $L$ stays at least 0.8 away from $-1$, a robust loop.

In [ ]:
plot_bode(S, margins=False, backend=BACKEND, title="S(s)")

**Complementary sensitivity $T$.** The mirror image of $S$: $|T| \approx 1$ at low frequency (the position follows the reference), a small peak near 3 rad/s, and a roll-off above it, where measurement noise is filtered out of the position.

In [ ]:
plot_bode(T, margins=False, backend=BACKEND, title="T(s)")

With $F = 1$ the two add up to one at every frequency, $S + T = \dfrac{1 + L}{1 + L} = 1$. This is the basic trade-off of feedback: $S$ and $T$ cannot both be small at the same frequency.

In [ ]:
w = np.logspace(-2, 3, 200)
_, S_jw = frequency_response(S, w=w)
_, T_jw = frequency_response(T, w=w)
np.allclose(S_jw + T_jw, 1.0)

**Load sensitivity $PS$.** A force disturbance $w$ enters at the plant input. At low frequency the integral action rejects it; at high frequency the mass's own inertia filters it. It is worst near 1.6 rad/s, close to the spring's natural frequency, where the plant itself amplifies most.

In [ ]:
plot_bode(PS, margins=False, backend=BACKEND, title="PS(s)")

**Noise sensitivity $CS$.** Measurement noise $v$ reaches the force command through the controller. At low frequency $CS \to 1/H(0) = k$: the controller pushes back exactly as hard as the spring. At high frequency $|CS| \to |C(j\omega)|$, and the derivative term makes $|C|$ large there: $K_p + K_d/\tau = 90$ N/m, about +39 dB. The position barely moves ($T$ is small up there), but the actuator chatters.

In [ ]:
plot_bode(CS, margins=False, backend=BACKEND, title="CS(s)")

## 3. A specification is one `bode` call

Specifications are written as bounds on these functions at given frequencies. `bode` takes the frequency in rad/s; a frequency $f$ in hertz is $\omega = 2\pi f$.

*The load force at 1 rad/s reaches the position attenuated below −20 dB:*

In [ ]:
_, PS_db, _ = bode(PS, w=LOAD_OMEGA)
PS_db[0] < -20.0

*Position noise at 50 rad/s reaches the force amplified by at most 20 dB:* this one fails, because of the derivative gain. A longer derivative filter $\tau$, or a filter $F$ on the measurement (`filter=F`), trades a little phase margin for a quieter actuator.

In [ ]:
_, CS_db, _ = bode(CS, w=NOISE_OMEGA)
CS_db[0] < 20.0

## 4. The same loop, simulated with a reference, a disturbance and noise

`closed_loop(C, plant, ...)` builds the loop that `C @ plant` builds, with optional boundary inputs: `r=True` (the reference, on by default), `w=True` (a force summed into the plant input) and `v=True` (noise summed into the measured position). Each extra input enters through a `Sum` block drawn with its signs, `+ in0` and `+ in1`. One more output exposes the force command $u$ next to the position $y$.

In [ ]:
loop = closed_loop(C, plant, w=True, v=True)  # inputs r, w, v
loop.connect_new_output_port("ctl", "u", "u")

In [ ]:
loop.plot_diagram()

Three source blocks drive the three inputs: a `Step` for the reference, $r(t) = 0.2$ m from $t = 1$ s, and two `Sine` sources, a load force $w(t) = 1.0 \sin(1.0\, t)$ N and a position noise $v(t) = 0.02 \sin(50\, t)$ m.

In [ ]:
diagram = DiagramSystem()
diagram.add_subsystem(Step(final_value=[REFERENCE], step_time=1.0), "reference")
diagram.add_subsystem(Sine(amplitude=LOAD_AMPLITUDE, omega=LOAD_OMEGA), "load")
diagram.add_subsystem(Sine(amplitude=NOISE_AMPLITUDE, omega=NOISE_OMEGA), "sensor_noise")
diagram.add_subsystem(loop, "loop")
diagram.connect("reference", "y", "loop", "r")
diagram.connect("load", "y", "loop", "w")
diagram.connect("sensor_noise", "y", "loop", "v")

In [ ]:
diagram.plot_diagram()

In [ ]:
traj = diagram.compute_trajectory(tf=20.0, dt=0.001, verbose=False)

In [ ]:
diagram.plot_trajectory(
    traj, signals=("reference:y", "load:y", "sensor_noise:y", "loop:y", "loop:u")
)

## 5. The prediction, checked

By superposition, in steady state the position is the reference (it is tracked exactly, $T(0) = 1$), plus the load sine scaled by $|PS(j\omega_w)|$, plus the noise sine scaled by $|T(j\omega_v)|$. The force carries the noise scaled by $|CS(j\omega_v)|$. The linear prediction of those two amplitudes:

In [ ]:
y_from_load = LOAD_AMPLITUDE * 10 ** (PS_db[0] / 20)  # m
u_from_noise = NOISE_AMPLITUDE * 10 ** (CS_db[0] / 20)  # N
y_from_load, u_from_noise

The simulation agrees: after the transient, the position settles on $r = 0.2$ m and swings by ±0.089 m at 1 rad/s (the noise adds only $0.02 \cdot |T(j50)| \approx 0.0007$ m), and the force carries a ±1.7 N chatter at 50 rad/s on top of the slow load compensation. The spike at $t = 1$ s is the derivative kick: the step in $r$ passes through $K_d/\tau$ at once. Change `NOISE_OMEGA` or `tau` and watch $|CS|$, and the chatter, follow.

## 6. Recap

| question | minilink | the math |
| --- | --- | --- |
| what an output disturbance does to $y$ | `sensitivity(plant=H, controller=C)` | $S = 1/(1 + L)$ |
| how $y$ follows $r$ | `complementary_sensitivity(...)` | $T = CH/(1 + L)$, $S + T = 1$ |
| what a load force does to $y$ | `load_sensitivity(...)` | $PS = H/(1 + L)$ |
| what sensor noise does to $u$ | `noise_sensitivity(...)` | $CS = CF/(1 + L)$ |
| a specification at one frequency | `bode(G, w=2 * np.pi * f)` | $20 \log_{10} \lvert G(j\omega) \rvert$ |
| the loop with reference, disturbance and noise inputs | `closed_loop(C, plant, w=True, v=True)` | $u_{plant} = u + w$, $y_{meas} = y + v$ |
| a signal into a port | `Step`, `Sine(amplitude, omega)` + `diagram.connect` | $a \sin(\omega t + \varphi)$ |